# C5 — CatBoost Tuning & Evaluation

CatBoost is the next tabular-model challenger after Logistic Regression,
Gradient Boosting, XGBoost and LightGBM.

## Goal

Evaluate whether CatBoost can improve the current best XGBoost/LightGBM results
on the same leakage-safe C4 split.

Metrics:

- PR-AUC ↑
- ROC-AUC ↑
- Log Loss ↓
- Brier Score ↓

Primary model-selection metric: **validation PR-AUC**, with the other three
metrics used as secondary probability-quality/discrimination checks.

## Data protocol

- Source: `data/c4_splits/train.csv`, `validation.csv`, `test.csv`
- Training: C4 TRAIN only
- CV: 5-fold `StratifiedGroupKFold`
- Group: `case_id`
- Validation: model comparison / selection
- Test: final untouched evaluation

CatBoost is given categorical columns natively; they are NOT one-hot encoded.


In [1]:

# ============================================================
# 0. SETUP
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import optuna
import joblib

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
if not (ROOT / "data" / "c4_splits").exists():
    ROOT = ROOT.parent

SPLIT_DIR = ROOT / "data" / "c4_splits"
REPORT_DIR = ROOT / "reports" / "c5_catboost"
MODEL_DIR = ROOT / "models" / "c5"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20260824
TARGET = "y"
GROUP_COL = "case_id"

N_SPLITS = 5
N_TRIALS = 40

train = pd.read_csv(
    SPLIT_DIR / "train.csv",
    low_memory=False,
)
valid = pd.read_csv(
    SPLIT_DIR / "validation.csv",
    low_memory=False,
)
test = pd.read_csv(
    SPLIT_DIR / "test.csv",
    low_memory=False,
)

for name, df in [
    ("train", train),
    ("validation", valid),
    ("test", test),
]:
    if TARGET not in df.columns:
        raise ValueError(f"{name} missing target `{TARGET}`.")
    if GROUP_COL not in df.columns:
        raise ValueError(f"{name} missing `{GROUP_COL}`.")
    y = pd.to_numeric(df[TARGET], errors="coerce")
    if y.isna().any() or not y.isin([0, 1]).all():
        raise ValueError(f"{name} target must be binary 0/1.")

print("Train:", train.shape)
print("Validation:", valid.shape)
print("Test:", test.shape)
print("Positive rates:")
print("Train:", train[TARGET].mean())
print("Validation:", valid[TARGET].mean())
print("Test:", test[TARGET].mean())


Train: (110385, 37)
Validation: (23471, 37)
Test: (23567, 37)
Positive rates:
Train: 0.2282194138696381
Validation: 0.22994333432746794
Test: 0.22590911019646115


## 1. Feature Contract

In [2]:

ID_COLUMNS = {
    "customer_id",
    "subscription_id",
    "case_id",
}

POLICY_COLUMNS = {
    "logging_policy",
}

INTERVENTION_COLUMNS = {
    "action_type",
    "channel",
}

FEATURES = [
    c for c in train.columns
    if c not in ID_COLUMNS | POLICY_COLUMNS | {TARGET}
]

CATEGORICAL_COLS = [
    c for c in FEATURES
    if (
        c in INTERVENTION_COLUMNS
        or not pd.api.types.is_numeric_dtype(train[c])
    )
]

NUMERIC_COLS = [
    c for c in FEATURES
    if (
        c not in CATEGORICAL_COLS
        and pd.api.types.is_numeric_dtype(train[c])
    )
]

print("Predictive features:", len(FEATURES))
print("Numeric:", len(NUMERIC_COLS))
print("Categorical:", len(CATEGORICAL_COLS))
print("Categorical columns:")
print(CATEGORICAL_COLS)

assert not (set(FEATURES) & ID_COLUMNS)
assert not (set(FEATURES) & POLICY_COLUMNS)


Predictive features: 32
Numeric: 20
Categorical: 12
Categorical columns:
['rail', 'bank_name', 'segment', 'plan_tier', 'value_tier', 'first_failure_class', 'failure_class_at_decision', 'age_band', 'state', 'locale_pref', 'action_type', 'channel']


## 2. Prepare CatBoost Inputs

CatBoost handles categorical features natively.

For CatBoost, missing categorical values are converted to a stable string token.
Numeric missing values are left as NaN so CatBoost can handle them natively.


In [3]:

def prepare_catboost_frame(df):
    X = df[FEATURES].copy()

    for col in CATEGORICAL_COLS:
        X[col] = (
            X[col]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    return X


X_train = prepare_catboost_frame(train)
X_valid = prepare_catboost_frame(valid)
X_test = prepare_catboost_frame(test)

y_train = train[TARGET].astype(int)
y_valid = valid[TARGET].astype(int)
y_test = test[TARGET].astype(int)

cat_feature_indices = [
    FEATURES.index(c)
    for c in CATEGORICAL_COLS
]

print("CatBoost categorical feature indices:")
print(cat_feature_indices)


CatBoost categorical feature indices:
[16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 30, 31]


## 3. Evaluation Metrics

In [4]:

def evaluate(y_true, proba):
    proba = np.clip(
        np.asarray(proba, dtype=float),
        1e-7,
        1 - 1e-7,
    )
    y_true = np.asarray(y_true)

    return {
        "pr_auc": float(
            average_precision_score(
                y_true,
                proba,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                proba,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                proba,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                proba,
            )
        ),
    }


def threshold_metrics(y_true, proba, threshold):
    pred = (
        np.asarray(proba) >= threshold
    ).astype(int)

    return {
        "threshold": float(threshold),
        "accuracy": float(
            (pred == np.asarray(y_true)).mean()
        ),
        "precision": float(
            precision_score(
                y_true,
                pred,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                pred,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                pred,
                zero_division=0,
            )
        ),
        "predicted_positive_rate": float(
            pred.mean()
        ),
    }


## 4. CatBoost Search Space

This is intentionally bounded. The aim is to test whether CatBoost can beat the
current XGBoost/LightGBM baselines, not to spend excessive compute blindly.

Primary tuning metric: CV PR-AUC.


In [5]:

def catboost_params(trial):
    return {
        "iterations": trial.suggest_int(
            "iterations",
            300,
            1200,
            step=100,
        ),
        "depth": trial.suggest_int(
            "depth",
            4,
            10,
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.12,
            log=True,
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1.0,
            30.0,
            log=True,
        ),
        "random_strength": trial.suggest_float(
            "random_strength",
            0.0,
            3.0,
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0.0,
            3.0,
        ),
        "border_count": trial.suggest_int(
            "border_count",
            64,
            255,
            step=32,
        ),
    }


## 5. 5-Fold Stratified Grouped CV

The same `case_id` must not appear in both training and validation portions of
a CV fold.

In [6]:

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

groups = train[GROUP_COL]


def catboost_objective(trial):
    params = catboost_params(trial)
    fold_scores = []
    fold_metrics = []

    for fold, (tr_idx, va_idx) in enumerate(
        cv.split(
            X_train,
            y_train,
            groups,
        )
    ):
        X_tr = X_train.iloc[tr_idx].copy()
        X_va = X_train.iloc[va_idx].copy()

        y_tr = y_train.iloc[tr_idx]
        y_va = y_train.iloc[va_idx]

        model = CatBoostClassifier(
            **params,
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=SEED + fold,
            thread_count=-1,
            verbose=False,
            allow_writing_files=False,
        )

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_feature_indices,
            eval_set=(X_va, y_va),
            early_stopping_rounds=60,
            verbose=False,
        )

        proba = model.predict_proba(
            X_va
        )[:, 1]

        metrics = evaluate(
            y_va,
            proba,
        )

        fold_metrics.append(metrics)
        fold_scores.append(
            metrics["pr_auc"]
        )

        trial.report(
            float(np.mean(fold_scores)),
            step=fold,
        )

        if trial.should_prune():
            raise optuna.TrialPruned()

    mean_metrics = {
        metric: float(
            np.mean(
                [
                    m[metric]
                    for m in fold_metrics
                ]
            )
        )
        for metric in [
            "pr_auc",
            "roc_auc",
            "log_loss",
            "brier",
        ]
    }

    for metric, value in mean_metrics.items():
        trial.set_user_attr(
            f"cv_{metric}",
            value,
        )

    return mean_metrics["pr_auc"]


## 6. Run Optuna

In [7]:

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=SEED,
    ),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=8,
        n_warmup_steps=2,
    ),
)

study.optimize(
    catboost_objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

best_trial = study.best_trial

print(
    "Best CV PR-AUC:",
    best_trial.value,
)

print("\nBest CV metrics:")
print(
    "PR-AUC :",
    best_trial.user_attrs["cv_pr_auc"],
)
print(
    "ROC-AUC:",
    best_trial.user_attrs["cv_roc_auc"],
)
print(
    "Log Loss:",
    best_trial.user_attrs["cv_log_loss"],
)
print(
    "Brier:",
    best_trial.user_attrs["cv_brier"],
)

print("\nBest parameters:")
print(best_trial.params)


  0%|          | 0/40 [00:00<?, ?it/s]

[W 2026-08-25 00:38:03,768] Trial 10 failed with parameters: {'iterations': 700, 'depth': 10, 'learning_rate': 0.020081196988323596, 'l2_leaf_reg': 1.104247688226757, 'random_strength': 2.9833664920548166, 'bagging_temperature': 0.18071525601761473, 'border_count': 224} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Soham\AppData\Local\Temp\ipykernel_31740\852661994.py", line 38, in catboost_objective
    model.fit(
  File "c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\catboost\core.py", line 5547, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline, use_best_model,
  File "c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\catboost\core.py", line 2716, in _fit
    self._train(
  File "c

KeyboardInterrupt: 

## 7. Optuna Diagnostics

In [ ]:

trials = study.trials_dataframe()

display(
    trials[
        [
            "number",
            "value",
            "state",
        ]
    ]
    .sort_values(
        "value",
        ascending=False,
    )
    .head(15)
)

plt.figure(
    figsize=(11, 5)
)

sns.lineplot(
    data=trials.sort_values("number"),
    x="number",
    y="value",
    marker="o",
)

plt.title(
    "CatBoost Optuna CV PR-AUC"
)

plt.xlabel("Trial")
plt.ylabel("CV PR-AUC")

plt.show()


## 8. Final CatBoost Training

Fit the selected configuration on the complete C4 training set, then evaluate
on validation and test.

In [ ]:

best_catboost = CatBoostClassifier(
    **best_trial.params,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=SEED,
    thread_count=-1,
    verbose=False,
    allow_writing_files=False,
)

best_catboost.fit(
    X_train,
    y_train,
    cat_features=cat_feature_indices,
    verbose=False,
)

train_proba = best_catboost.predict_proba(
    X_train
)[:, 1]

valid_proba = best_catboost.predict_proba(
    X_valid
)[:, 1]

test_proba = best_catboost.predict_proba(
    X_test
)[:, 1]

catboost_results = pd.DataFrame([
    {
        "model": "catboost_tuned",
        "split": "train",
        **evaluate(
            y_train,
            train_proba,
        ),
    },
    {
        "model": "catboost_tuned",
        "split": "validation",
        **evaluate(
            y_valid,
            valid_proba,
        ),
    },
    {
        "model": "catboost_tuned",
        "split": "test",
        **evaluate(
            y_test,
            test_proba,
        ),
    },
])

display(
    catboost_results
)


## 9. Compare Against Existing C5 Baselines

In [ ]:

baseline_path = (
    ROOT
    / "reports"
    / "c5_baseline"
    / "baseline_metrics.csv"
)

comparison_frames = []

if baseline_path.exists():
    baseline = pd.read_csv(
        baseline_path
    )

    comparison_frames.append(
        baseline[
            [
                "model",
                "split",
                "pr_auc",
                "roc_auc",
                "log_loss",
                "brier",
            ]
        ]
    )

# Existing tuned model reports, if available.
xgb_path = (
    ROOT
    / "reports"
    / "c5_xgboost"
    / "metrics.csv"
)

if xgb_path.exists():
    xgb = pd.read_csv(xgb_path)

    rename_map = {
        "model": "model",
        "split": "split",
        "pr_auc": "pr_auc",
        "roc_auc": "roc_auc",
        "log_loss": "log_loss",
        "brier": "brier",
    }

    xgb = xgb[
        [
            c for c in rename_map
            if c in xgb.columns
        ]
    ]

    comparison_frames.append(
        xgb
    )

comparison_frames.append(
    catboost_results[
        [
            "model",
            "split",
            "pr_auc",
            "roc_auc",
            "log_loss",
            "brier",
        ]
    ]
)

comparison = pd.concat(
    comparison_frames,
    ignore_index=True,
)

display(
    comparison.sort_values(
        [
            "split",
            "pr_auc",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


## 10. Validation Threshold Analysis

The probability model is evaluated separately from the hard 0/1 threshold.

Threshold selection is performed on validation only.

In [ ]:

thresholds = np.arange(
    0.05,
    0.951,
    0.005,
)

threshold_rows = [
    threshold_metrics(
        y_valid,
        valid_proba,
        float(t),
    )
    for t in thresholds
]

threshold_results = pd.DataFrame(
    threshold_rows
)

display(
    threshold_results
    .sort_values(
        "accuracy",
        ascending=False,
    )
    .head(15)
)

plt.figure(
    figsize=(11, 6)
)

sns.lineplot(
    data=threshold_results,
    x="threshold",
    y="accuracy",
    label="Accuracy",
)

sns.lineplot(
    data=threshold_results,
    x="threshold",
    y="precision",
    label="Precision",
)

sns.lineplot(
    data=threshold_results,
    x="threshold",
    y="recall",
    label="Recall",
)

sns.lineplot(
    data=threshold_results,
    x="threshold",
    y="f1",
    label="F1",
)

plt.ylim(0, 1)
plt.title("CatBoost Validation Threshold Trade-off")
plt.xlabel("Threshold")
plt.ylabel("Metric")
plt.show()


## 11. Save C5 CatBoost Artifacts

In [ ]:

catboost_results.to_csv(
    REPORT_DIR / "catboost_metrics.csv",
    index=False,
)

comparison.to_csv(
    REPORT_DIR / "model_comparison.csv",
    index=False,
)

threshold_results.to_csv(
    REPORT_DIR / "catboost_validation_thresholds.csv",
    index=False,
)

study.trials_dataframe().to_csv(
    REPORT_DIR / "catboost_optuna_trials.csv",
    index=False,
)

with open(
    REPORT_DIR / "catboost_best_params.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "optimization_metric": "pr_auc",
            "cv": "5-fold StratifiedGroupKFold",
            "group": GROUP_COL,
            "best_cv_metrics": {
                "pr_auc":
                    best_trial.user_attrs["cv_pr_auc"],
                "roc_auc":
                    best_trial.user_attrs["cv_roc_auc"],
                "log_loss":
                    best_trial.user_attrs["cv_log_loss"],
                "brier":
                    best_trial.user_attrs["cv_brier"],
            },
            "best_params":
                best_trial.params,
        },
        f,
        indent=2,
    )

joblib.dump(
    {
        "model": best_catboost,
        "features": FEATURES,
        "categorical_features": CATEGORICAL_COLS,
    },
    MODEL_DIR / "catboost_tuned_c5.joblib",
)

print(
    "CatBoost C5 artifacts saved."
)
